# 02 · Tutelas — Calidad y Normalización
**Objetivo:** Limpiar y validar el dataset para que el análisis del notebook 03 sea confiable.

Regla: **no se elimina ningún registro original**. Se agregan columnas de calidad y se documentan los problemas como hallazgos.

**Prerequisito:** haber corrido `01_tutelas_exploration.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

DATA_PATH = '../data/tutelas - Detalle1.csv'
REFERENCE_DATE = pd.Timestamp('2026-05-11')
MIN_REGISTROS_POR_YEAR = 500  # consistente con nb01

# Columnas completamente vacías en 2026 — eliminadas en nb01, se repite aquí
COLS_VACIAS = [
    'Fecha y hora fallo Corte Constitucional',
    'Fecha vencimiento fallo corte',
    'PBS/NO PBS',
    'Area causal_4',
    'Tutela evitable',
    'Motivo Causal Juridico',
    'Prestación sucesiva 1ra instancia',
]


## 1. Carga y parseo de fechas

In [2]:
df_raw = pd.read_csv(DATA_PATH, encoding='utf-8', sep=None, engine='python')

# ── Fechas (excluye las dos de Corte Constitucional, que son 100% nulas) ──
date_cols = [
    'Fecha de creación',
    'Fecha y hora notificación tutela',
    'Fecha Vencimiento Admision',
    'Fecha Reasignacion Concepto Salud',
    'Fecha_Asignacion_Concepto',
    'Fecha Limite Concepto Admisión',
    'Fecha finalización concepto',
    'Fecha y hora de Contestación',
    'Fecha y hora notificación 1ra Instancia',
    'Fecha vencimiento fallo 1ra Instancia',
    'Fecha y hora Notificación 2da Instancia',
    'Fecha vencimiento fallo 2da Instancia',
]

for col in date_cols:
    df_raw[col] = pd.to_datetime(df_raw[col], errors='coerce')

# ── Filtro de año (igual que nb01: conservar solo años con ≥500 registros) ──
df_raw['_year'] = df_raw['Fecha y hora notificación tutela'].dt.year
year_counts = df_raw['_year'].value_counts()
años_validos = year_counts[year_counts >= MIN_REGISTROS_POR_YEAR].index

descartados = df_raw[~df_raw['_year'].isin(años_validos)]
df_filtered = df_raw[df_raw['_year'].isin(años_validos)].copy()

print(f'Registros totales en CSV:  {len(df_raw):,}')
print(f'Registros descartados:     {len(descartados):,}  (años: {sorted(descartados["_year"].dropna().unique().astype(int).tolist())})')
print(f'Registros en df_filtered:  {len(df_filtered):,}  (año: {sorted(df_filtered["_year"].dropna().unique().astype(int).tolist())})')

# ── Eliminar columnas completamente vacías ──
cols_a_drop = [c for c in COLS_VACIAS if c in df_filtered.columns]
df_filtered.drop(columns=cols_a_drop, inplace=True)
df_filtered.drop(columns=['_year'], inplace=True)

print(f'\nColumnas eliminadas ({len(cols_a_drop)}): {cols_a_drop}')
print(f'Shape final: {df_filtered.shape}')


Registros totales en CSV:  12,952
Registros descartados:     201  (años: [2013, 2025])
Registros en df_filtered:  12,751  (año: [2026])

Columnas eliminadas (7): ['Fecha y hora fallo Corte Constitucional', 'Fecha vencimiento fallo corte', 'PBS/NO PBS', 'Area causal_4', 'Tutela evitable', 'Motivo Causal Juridico', 'Prestación sucesiva 1ra instancia']
Shape final: (12751, 40)


## 2. Normalización de texto en campos categóricos

In [3]:
# Estandarizar texto: mayúsculas, sin espacios extra
# (excluye Motivo Causal Juridico, PBS/NO PBS y Prestación sucesiva — eliminadas en carga)
text_cols = [
    'Estado del ciclo de vida',
    'Compañía',
    'Regional T',
    'Area causal',
    'Clasificación fallo 1ra Instancia',
    'Clasificación fallo 2da Instancia',
    'Tipos respuesta',
    'Medida provisional',
    'Tipo de Prestación',
]

for col in text_cols:
    if col in df_filtered.columns:
        df_filtered[col] = df_filtered[col].astype(str).str.strip().str.upper()
        df_filtered[col] = df_filtered[col].replace({'NAN': np.nan, 'NONE': np.nan})

print('Normalización de texto aplicada.')
print(f'Columnas normalizadas: {[c for c in text_cols if c in df_filtered.columns]}')


Normalización de texto aplicada.
Columnas normalizadas: ['Estado del ciclo de vida', 'Compañía', 'Regional T', 'Area causal', 'Clasificación fallo 1ra Instancia', 'Clasificación fallo 2da Instancia', 'Tipos respuesta', 'Medida provisional', 'Tipo de Prestación']


## 3. Detección de duplicados

In [4]:
# Duplicados por nombre de fichero (ID principal)
dup_fichero = df_filtered['Nombre del fichero'].duplicated(keep=False)
print(f'Registros con Nombre del fichero duplicado: {dup_fichero.sum()}')

# Duplicados exactos (todas las columnas)
dup_total = df_filtered.duplicated(keep=False)
print(f'Registros completamente duplicados: {dup_total.sum()}')

if dup_fichero.sum() > 0:
    print('\nEjemplos de ficheros duplicados:')
    print(df_filtered[dup_fichero][['Nombre del fichero', 'Estado del ciclo de vida', 'Fecha de creación']].head(6).to_string())


Registros con Nombre del fichero duplicado: 2
Registros completamente duplicados: 0

Ejemplos de ficheros duplicados:
                               Nombre del fichero Estado del ciclo de vida   Fecha de creación
2890  EXP-T-26-000002891 - 2026-0026 - 1040180343                  NULIDAD 2026-01-26 11:49:45
9664  EXP-T-26-000002891 - 2026-0026 - 1040180343     GESTIÓN FALLO 1 ÁREA 2026-03-09 10:42:59


## 4. Validación de coherencia temporal

In [5]:
inconsistencias = pd.DataFrame(index=df_filtered.index)

# Contestación antes de notificación (imposible)
inconsistencias['contestacion_antes_notif'] = (
    df_filtered['Fecha y hora de Contestación'] < df_filtered['Fecha y hora notificación tutela']
).fillna(False)

# Vencimiento admisión antes de notificación
inconsistencias['vencimiento_antes_notif'] = (
    df_filtered['Fecha Vencimiento Admision'] < df_filtered['Fecha y hora notificación tutela']
).fillna(False)

# Fallo 1ra instancia con fecha futura (posterior a fecha de análisis)
inconsistencias['fallo1_futuro'] = (
    df_filtered['Fecha vencimiento fallo 1ra Instancia'] > REFERENCE_DATE
).fillna(False)

# Contestación posterior al vencimiento (respuesta tardía — hallazgo, no inconsistencia)
inconsistencias['contestacion_tardia'] = (
    df_filtered['Fecha y hora de Contestación'] > df_filtered['Fecha Vencimiento Admision']
).fillna(False)

print('Resumen de validaciones temporales:')
for col in inconsistencias.columns:
    n = inconsistencias[col].sum()
    pct = n / len(df_filtered) * 100
    flag = '  ← INCONSISTENCIA' if 'tardia' not in col else '  ← hallazgo (respuesta tardía)'
    print(f'  {col}: {n:,} ({pct:.1f}%){flag}')


Resumen de validaciones temporales:
  contestacion_antes_notif: 4 (0.0%)  ← INCONSISTENCIA
  vencimiento_antes_notif: 14 (0.1%)  ← INCONSISTENCIA
  fallo1_futuro: 6 (0.0%)  ← INCONSISTENCIA
  contestacion_tardia: 1,110 (8.7%)  ← hallazgo (respuesta tardía)


## 5. Columna de calidad de dato por registro

In [6]:
# Campos mínimos obligatorios para el análisis
# Nota: Area causal tiene 54% nulos en EPS (patrón estructural, no error) → no se penaliza aquí
campos_criticos = [
    'Fecha y hora notificación tutela',
    'Fecha Vencimiento Admision',
    'Regional T',
    'Estado del ciclo de vida',
]

nulos_criticos = df_filtered[campos_criticos].isnull().sum(axis=1)

df_filtered['calidad_dato'] = np.where(
    inconsistencias['contestacion_antes_notif'] | inconsistencias['vencimiento_antes_notif'],
    'Inconsistente',
    np.where(nulos_criticos > 0, 'Incompleto', 'OK')
)

print('Distribución calidad_dato:')
print(df_filtered['calidad_dato'].value_counts())
print(f'\n% registros confiables (OK): {(df_filtered["calidad_dato"] == "OK").mean()*100:.1f}%')


Distribución calidad_dato:
calidad_dato
OK               12726
Inconsistente       18
Incompleto           7
Name: count, dtype: int64

% registros confiables (OK): 99.8%


## 6. Exportar dataset limpio

In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)

df_filtered.to_csv('../data/processed/tutelas_clean.csv', index=False, encoding='utf-8')
print(f'Guardado: ../data/processed/tutelas_clean.csv')
print(f'  Filas:    {len(df_filtered):,}')
print(f'  Columnas: {len(df_filtered.columns)}')
print(f'\nColumna nueva añadida:')
print(f'  + calidad_dato  →  {df_filtered["calidad_dato"].value_counts().to_dict()}')


Guardado: ../data/processed/tutelas_clean.csv
  Filas:    12,751
  Columnas: 41

Columna nueva añadida:
  + calidad_dato  →  {'OK': 12726, 'Inconsistente': 18, 'Incompleto': 7}


## 7. Resumen de hallazgos de calidad


### 7a. % de registros por estado de calidad


In [8]:
total = len(df_filtered)
resumen_calidad = (
    df_filtered['calidad_dato']
    .value_counts()
    .rename_axis('Estado')
    .reset_index(name='Registros')
)
resumen_calidad['%'] = (resumen_calidad['Registros'] / total * 100).round(2)

print(f'Dataset: {total:,} registros (2026)\n')
print(resumen_calidad.to_string(index=False))


Dataset: 12,751 registros (2026)

       Estado  Registros     %
           OK      12726 99.80
Inconsistente         18  0.14
   Incompleto          7  0.05


### 7b. Campos con mayor tasa de nulos


In [9]:
nulos_resumen = (
    df_filtered.isnull()
    .mean()
    .mul(100)
    .round(1)
    .rename('% nulo')
    .reset_index()
    .rename(columns={'index': 'Columna'})
    .query('`% nulo` > 0')
    .sort_values('% nulo', ascending=False)
    .reset_index(drop=True)
)

# Clasificar impacto para el informe
def clasificar(pct):
    if pct >= 80:
        return 'Estructural / no aplica al proceso'
    elif pct >= 30:
        return 'Alto — limita análisis'
    elif pct >= 5:
        return 'Moderado — tratar con precaución'
    else:
        return 'Bajo — manejable'

nulos_resumen['Impacto'] = nulos_resumen['% nulo'].apply(clasificar)

print('Campos con nulos en df_filtered (2026):\n')
print(nulos_resumen.to_string(index=False))


Campos con nulos en df_filtered (2026):

                                Columna  % nulo                            Impacto
                          Area causal_3    99.7 Estructural / no aplica al proceso
                            Odontologia    99.4 Estructural / no aplica al proceso
                          Area causal_2    94.0 Estructural / no aplica al proceso
      Clasificación fallo 2da Instancia    87.5 Estructural / no aplica al proceso
  Horas para cumplimiento 2da Instancia    86.9 Estructural / no aplica al proceso
  Fecha vencimiento fallo 2da Instancia    86.9 Estructural / no aplica al proceso
Fecha y hora Notificación 2da Instancia    86.9 Estructural / no aplica al proceso
         Usuario área técnica ARL SALUD    85.2 Estructural / no aplica al proceso
              Tiene Fallo 2da instancia    56.4             Alto — limita análisis
                            Area causal    54.3             Alto — limita análisis
      Fecha Reasignacion Concepto Salud    43.

### 7c. Inconsistencias temporales detectadas


In [10]:
descripciones = {
    'contestacion_antes_notif': 'Contestación con fecha ANTERIOR a la notificación (imposible)',
    'vencimiento_antes_notif':  'Vencimiento de admisión ANTERIOR a la notificación (imposible)',
    'fallo1_futuro':            'Vencimiento fallo 1ra instancia posterior a fecha de análisis',
    'contestacion_tardia':      'Contestación POSTERIOR al vencimiento de admisión (respuesta tardía)',
}

print(f'{"Tipo de inconsistencia":<60}  {"N":>6}  {"% ":>6}  {"Tipo"}')
print('─' * 90)
for col, desc in descripciones.items():
    n = inconsistencias[col].sum()
    pct = n / total * 100
    tipo = 'Hallazgo operativo' if 'tardia' in col else 'Inconsistencia de dato'
    print(f'{desc:<60}  {n:>6,}  {pct:>5.1f}%  {tipo}')

print()
# Detalle de los 18 registros inconsistentes
inconsistentes_idx = inconsistencias[
    inconsistencias['contestacion_antes_notif'] | inconsistencias['vencimiento_antes_notif']
].index
print(f'Registros marcados como Inconsistente: {len(inconsistentes_idx)}')
print(df_filtered.loc[inconsistentes_idx, [
    'Nombre del fichero',
    'Fecha y hora notificación tutela',
    'Fecha Vencimiento Admision',
    'Fecha y hora de Contestación',
    'Regional T',
    'calidad_dato'
]].to_string())


Tipo de inconsistencia                                             N      %   Tipo
──────────────────────────────────────────────────────────────────────────────────────────
Contestación con fecha ANTERIOR a la notificación (imposible)       4    0.0%  Inconsistencia de dato
Vencimiento de admisión ANTERIOR a la notificación (imposible)      14    0.1%  Inconsistencia de dato
Vencimiento fallo 1ra instancia posterior a fecha de análisis       6    0.0%  Inconsistencia de dato
Contestación POSTERIOR al vencimiento de admisión (respuesta tardía)   1,110    8.7%  Hallazgo operativo

Registros marcados como Inconsistente: 18
                                      Nombre del fichero Fecha y hora notificación tutela Fecha Vencimiento Admision Fecha y hora de Contestación         Regional T   calidad_dato
30           EXP-T-26-000000030 - 2025-0584 - 1002202538              2026-01-02 11:26:00        2025-12-19 17:00:00          2026-01-15 16:12:04              NORTE  Inconsistente
332        

### 7d. Riesgos de calidad por regional


In [11]:
import matplotlib.pyplot as plt

# ── Calidad por regional ──────────────────────────────────────────────────
calidad_regional = (
    df_filtered.groupby('Regional T')['calidad_dato']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .unstack(fill_value=0)
    .assign(total=df_filtered.groupby('Regional T').size())
    .sort_values('total', ascending=False)
)

print('Calidad de dato por regional (%):\n')
print(calidad_regional.to_string())

# ── Tasa de nulos en campos clave por regional ────────────────────────────
campos_nulo_analisis = [
    'Area causal',
    'Clasificación fallo 1ra Instancia',
    'Fecha y hora de Contestación',
    'Medida provisional',
]
print('\nTasa de nulos en campos clave por regional (%):\n')
nulos_regional = (
    df_filtered.groupby('Regional T')[campos_nulo_analisis]
    .apply(lambda g: g.isnull().mean() * 100)
    .round(1)
)
print(nulos_regional.to_string())

# ── Respuestas tardías por regional ──────────────────────────────────────
df_con_contestacion = df_filtered[df_filtered['Fecha y hora de Contestación'].notna()].copy()
tardias_regional = (
    inconsistencias.loc[df_con_contestacion.index, 'contestacion_tardia']
    .groupby(df_filtered.loc[df_con_contestacion.index, 'Regional T'])
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'tardias', 'count': 'con_contestacion'})
)
tardias_regional['% tardías'] = (tardias_regional['tardias'] / tardias_regional['con_contestacion'] * 100).round(1)
tardias_regional = tardias_regional.sort_values('% tardías', ascending=False)

print('\nRespuestas tardías (contestación > vencimiento admisión) por regional:\n')
print(tardias_regional.to_string())


Calidad de dato por regional (%):

calidad_dato                                 Incompleto  Inconsistente     OK  total
Regional T                                                                          
ANTIOQUIA                                           0.0            0.0   99.9   6071
NORTE                                               0.0            0.5   99.5   1948
CENTRO                                              0.1            0.2   99.8   1705
OCCIDENTE                                           0.0            0.1   99.9   1579
EJE CAFETERO                                        0.1            0.0   99.9   1380
NORTE - SANTANDER                                   0.0            1.6   98.4     64
NORTE - ATLANTICO/SAN ANDRES  Y PROVIDENCIA         0.0            0.0  100.0      1
NORTE - BOLIVAR/ GUAJIRA                            0.0            0.0  100.0      1
NORTE - SUCRE/MAGDALENA                             0.0            0.0  100.0      1

Tasa de nulos en campos clave

In [12]:
# ── Conclusiones consolidadas para el informe ejecutivo ──────────────────

print('=' * 70)
print('  HALLAZGOS DE CALIDAD — TUTELAS 2026  ')
print('=' * 70)

# 7a
ok_pct = (df_filtered['calidad_dato'] == 'OK').mean() * 100
inc_pct = (df_filtered['calidad_dato'] == 'Inconsistente').mean() * 100
incomp_pct = (df_filtered['calidad_dato'] == 'Incompleto').mean() * 100
print(f'\n1. CONFIABILIDAD DEL DATASET')
print(f'   • {ok_pct:.1f}% de registros son confiables (OK)')
print(f'   • {inc_pct:.2f}% tienen inconsistencias temporales (18 registros)')
print(f'   • {incomp_pct:.2f}% tienen campos críticos vacíos (7 registros)')
print(f'   → El dataset es apto para análisis sin imputación de datos.')

# 7b
campos_impacto_alto = nulos_resumen[nulos_resumen['Impacto'] == 'Alto — limita análisis']['Columna'].tolist()
campos_estructurales = nulos_resumen[nulos_resumen['Impacto'] == 'Estructural / no aplica al proceso']['Columna'].tolist()
print(f'\n2. CAMPOS CON NULOS DE IMPACTO')
print(f'   Estructurales (proceso no llega): {len(campos_estructurales)} columnas (2da instancia, etc.)')
print(f'   Alto impacto: {len(campos_impacto_alto)} columnas — {campos_impacto_alto}')
print(f'   Nota: "Area causal" tiene 54% de nulos, concentrados en EPS (68.6%).')
print(f'   Es un patrón operativo, no un error de ingreso.')

# 7c
n_tardias = inconsistencias['contestacion_tardia'].sum()
pct_tardias = n_tardias / total * 100
print(f'\n3. INCONSISTENCIAS TEMPORALES')
print(f'   • 4 contestaciones antes de la notificación → error de ingreso')
print(f'   • 14 vencimientos antes de la notificación → error de ingreso')
print(f'   • {n_tardias:,} respuestas tardías ({pct_tardias:.1f}%) → KPI operativo para nb03')

# 7d — calcular la regional con más riesgo
tardias_top = tardias_regional['% tardías'].idxmax()
tardias_top_pct = tardias_regional.loc[tardias_top, '% tardías']
print(f'\n4. RIESGO POR REGIONAL')
print(f'   • Regional con mayor tasa de respuesta tardía: {tardias_top} ({tardias_top_pct}%)')
nulos_area_causal_regional = nulos_regional['Area causal'].sort_values(ascending=False)
top_nulo_regional = nulos_area_causal_regional.index[0]
print(f'   • Regional con mayor % de nulos en "Area causal": {top_nulo_regional} ({nulos_area_causal_regional.iloc[0]}%)')
print(f'   → Antioquia concentra ~47% del volumen total; sus métricas son representativas.')

print(f'\n> Dataset exportado: data/processed/tutelas_clean.csv  |  {total:,} filas × {len(df_filtered.columns)} cols')


  HALLAZGOS DE CALIDAD — TUTELAS 2026  

1. CONFIABILIDAD DEL DATASET
   • 99.8% de registros son confiables (OK)
   • 0.14% tienen inconsistencias temporales (18 registros)
   • 0.05% tienen campos críticos vacíos (7 registros)
   → El dataset es apto para análisis sin imputación de datos.

2. CAMPOS CON NULOS DE IMPACTO
   Estructurales (proceso no llega): 8 columnas (2da instancia, etc.)
   Alto impacto: 6 columnas — ['Tiene Fallo 2da instancia', 'Area causal', 'Fecha Reasignacion Concepto Salud', 'Usuario área técnica EPS SALUD', 'Area causal_Salud EPS', 'Tipo de Prestación']
   Nota: "Area causal" tiene 54% de nulos, concentrados en EPS (68.6%).
   Es un patrón operativo, no un error de ingreso.

3. INCONSISTENCIAS TEMPORALES
   • 4 contestaciones antes de la notificación → error de ingreso
   • 14 vencimientos antes de la notificación → error de ingreso
   • 1,110 respuestas tardías (8.7%) → KPI operativo para nb03

4. RIESGO POR REGIONAL
   • Regional con mayor tasa de respuesta